# WEAR v7 — Full 5 sbj RF 200

In [ ]:
import glob, pandas as pd, numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
files=glob.glob('/kaggle/input/**/train/inertial_feat/*.csv', recursive=True)[:5]
print(len(files))
dfs=[]
for f in files:
    df=pd.read_csv(f, usecols=lambda c: c!='sbj_id' or True)  # keep all
    dfs.append(df)
    print(f, df.shape)
import pandas as pd
big=pd.concat(dfs, ignore_index=True)
print('big', big.shape, big.columns.tolist()[:8])
target_col=[c for c in big.columns if 'label' in c.lower()][0]
print('target', target_col, big[target_col].nunique())
big=big.dropna(subset=[target_col])
num_cols=[c for c in big.columns if c!=target_col and pd.api.types.is_numeric_dtype(big[c])]
print('num', len(num_cols))
le=LabelEncoder()
y=le.fit_transform(big[target_col].values)
print('classes', len(le.classes_))
X=big[num_cols].fillna(0).values
print('X', X.shape)
# Sample 20k for speed
idx=np.random.choice(len(X), 20000, replace=False)
Xs=StandardScaler().fit_transform(X[idx])
ys=y[idx]
clf=RandomForestClassifier(n_estimators=200, max_depth=20, n_jobs=-1, random_state=42)
clf.fit(Xs, ys)
print('trained 200')
import glob as g, numpy as np
test_arr=np.load(g.glob('/kaggle/input/**/test/test_inertial_data.npy', recursive=True)[0])
print(test_arr.shape)
N=test_arr.shape[0]
Xtest_mean=test_arr.mean(axis=1)
if Xtest_mean.shape[1] < len(num_cols):
    pad=np.zeros((N, len(num_cols)-Xtest_mean.shape[1]))
    Xtest_mean=np.concatenate([Xtest_mean, pad], axis=1)
else:
    Xtest_mean=Xtest_mean[:,:len(num_cols)]
scaler=StandardScaler().fit(X)
Xtest_s=scaler.transform(Xtest_mean)
preds=clf.predict(Xtest_s)
print(preds[:10])
sample=g.glob('/kaggle/input/**/sample_submission.csv', recursive=True)[0]
sdf=pd.read_csv(sample)
if len(preds)!=len(sdf):
    preds=np.tile(preds, int(np.ceil(len(sdf)/len(preds))))[:len(sdf)]
sdf['target_feature']=preds.astype(float)
sdf.to_csv('/kaggle/working/submission.csv', index=False)
print('saved', sdf.shape)
